In [30]:
#Preparar el entorno de entrenamiento y empaquetado de modelos.
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier


In [33]:
import os

def find_project_root(marker_dirs=("artifacts", "data")):
    for root, dirs, files in os.walk("/", topdown=True):
        if all(m in dirs for m in marker_dirs):
            return root
    raise RuntimeError("No se encontró el project root con artifacts/ y data/")

print("Buscando project root... (puede tardar unos segundos)")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

print("Project root detectado:", os.getcwd())
print("Contenido:", os.listdir("."))


Buscando project root... (puede tardar unos segundos)
Project root detectado: /mnt/batch/tasks/shared/LS_root/mounts/clusters/testingnewmodels/code/Users/jaragono/tfmg10-credit_scoring
Contenido: ['.amlignore', '.amlignore.amltmp', '.git', '.gitignore', '.gitignore.amltmp', '.ipynb_checkpoints', 'artifacts', 'configs', 'data', 'FETCH_HEAD', 'notebooks', 'src']


In [34]:
#Se utiliza el mismo dataset base para todos los modelos, garantizando comparabilidad
df = pd.read_csv("data/dataset_final_modelo_10k.csv")

X = df.drop("pago", axis=1)
y = df["pago"]

print(df.shape)
print(y.value_counts())


(10000, 16)
0    6967
1    3033
Name: pago, dtype: int64


In [35]:
# Eliminar identificador personal (DUI)
df = df.drop(columns=["DUI_clean"])

# Definir variables predictoras y variable objetivo
X = df.drop("pago", axis=1)
y = df["pago"]

print("Shape del dataset:", df.shape)
print("Features usadas:", X.columns.tolist())


Shape del dataset: (10000, 15)
Features usadas: ['edad', 'ingresos_declarados', 'nivel_endeudamiento', 'utilizacion_tarjetas', 'score_buro', 'morosidad_prev', 'historial_empresa', 'consumo_electrico', 'pago_serv_publicos', 'titularidad_serv_publico', 'remesas', 'ingresos_bancarios', 'ultimo_consumo_movil', 'portabilidad']


In [36]:
#Split controlado
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)


In [37]:
#Registro local de modelos
models = {
    "xgb": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ),
    "lgbm": LGBMClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1 
    ),
    "gboost": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
}
#verbose=-1 elimina los warnings que imprime el modelo para mayor limpieza en los resultados

In [38]:
#Ignorar warnings
import warnings
warnings.filterwarnings("ignore")

#Loop de entrenamiento
results = {}

for name, model in models.items():
    print(f"Entrenando {name}...")
    
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        "roc_auc": roc_auc_score(y_test, y_pred_proba),
        "f1": f1_score(y_test, (y_pred_proba >= 0.5).astype(int))
    }

results


Entrenando xgb...
Entrenando lgbm...
Entrenando gboost...


{'xgb': {'roc_auc': 0.6768941584731059, 'f1': 0.256},
 'lgbm': {'roc_auc': 0.6559067248540933, 'f1': 0.23069936421435058},
 'gboost': {'roc_auc': 0.6796587622903412, 'f1': 0.31517183570829843}}

In [39]:
pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)


,roc_auc,f1
gboost,0.679659,0.315172
xgb,0.676894,0.256000
lgbm,0.655907,0.230699


In [40]:
timestamp = datetime.now().strftime("%Y%m%d")

for name, model in models.items():
    path = f"artifacts/credit_model_{name}_{timestamp}.pkl"
    joblib.dump(model, path)
    print(f"Modelo {name} guardado en {path}")


Modelo xgb guardado en artifacts/credit_model_xgb_20260113.pkl
Modelo lgbm guardado en artifacts/credit_model_lgbm_20260113.pkl
Modelo gboost guardado en artifacts/credit_model_gboost_20260113.pkl


In [41]:
from azureml.core import Workspace, Model

# Cargar workspace desde config.json
ws = Workspace.from_config()
# ---------- MODELO XGBoost ----------
xgb_model_path = "artifacts/credit_model_xgb_20260113.pkl"

xgb_registered = Model.register(
    workspace=ws,
    model_path=xgb_model_path,
    model_name="g1tfm_credit_scoring_model_xgb",
    description="Modelo XGBoost sin identificadores personales (DUI_clean) para scoring crediticio"
)

print("Modelo registrado:", xgb_registered.name)
print("Versión:", xgb_registered.version)

# ---------- MODELO LIGHTGBM ----------
lgbm_model_path = "artifacts/credit_model_lgbm_20260113.pkl"

lgbm_registered = Model.register(
    workspace=ws,
    model_path=lgbm_model_path,
    model_name="g1tfm_credit_scoring_model_lgbm",
    description="Modelo LightGBM sin identificadores personales (DUI_clean) para scoring crediticio"
)

print("Modelo registrado:", lgbm_registered.name)
print("Versión:", lgbm_registered.version)


# ---------- MODELO GRADIENT BOOSTING ----------
gboost_model_path = "artifacts/credit_model_gboost_20260113.pkl"

gboost_registered = Model.register(
    workspace=ws,
    model_path=gboost_model_path,
    model_name="g1tfm_credit_scoring_model_gboost",
    description="Modelo Gradient Boosting sin identificadores personales (DUI_clean) para scoring crediticio"
)

print("Modelo registrado:", gboost_registered.name)
print("Versión:", gboost_registered.version)


Registering model g1tfm_credit_scoring_model_xgb
Modelo registrado: g1tfm_credit_scoring_model_xgb
Versión: 2
Registering model g1tfm_credit_scoring_model_lgbm
Modelo registrado: g1tfm_credit_scoring_model_lgbm
Versión: 4
Registering model g1tfm_credit_scoring_model_gboost
Modelo registrado: g1tfm_credit_scoring_model_gboost
Versión: 3


In [42]:
Model(ws, name="g1tfm_credit_scoring_model_lgbm")


Model(workspace=Workspace.create(name='g10tfm-ml', subscription_id='506e4ca3-b9db-4960-aedf-c02bf0ea7e28', resource_group='jaragono-rg'), name=g1tfm_credit_scoring_model_lgbm, id=g1tfm_credit_scoring_model_lgbm:4, version=4, tags={}, properties={})

In [43]:
import joblib

xgb_test = joblib.load("artifacts/credit_model_xgb_20260113.pkl")

proba = xgb_test.predict_proba(X_test)
print(proba)


[[0.5546719  0.4453281 ]
 [0.8043146  0.19568536]
 [0.8805652  0.11943478]
 ...
 [0.75069207 0.24930795]
 [0.5427034  0.45729658]
 [0.9465398  0.05346021]]
